In [ ]:
import pandas as pd

path = r"data\2011\results.csv"

# Read first 5 rows, no header interpretation
raw = pd.read_csv(
    path,
    header=None,
    nrows=5,
    encoding="cp1252",
    dtype=str,
    keep_default_na=False
)

raw

In [ ]:
print("Shape:", raw.shape)
print("\n=== Row 0 (questions) ===")
for i, v in enumerate(raw.iloc[0]):
    if v.strip():
        print(f"  Col {i}: {v}")

print("\n=== Row 1 (options) ===")
for i, v in enumerate(raw.iloc[1]):
    if v.strip():
        print(f"  Col {i}: {v}")

print("\n=== Row 2 (first respondent) ===")
for i, v in enumerate(raw.iloc[2]):
    if v.strip():
        print(f"  Col {i}: {v}")

In [ ]:
raw_headers = pd.read_csv(
    path, header=None, nrows=2, encoding="cp1252",
    dtype=str, keep_default_na=False
)
raw_headers

In [ ]:
# Fill merged-question blanks across grouped option columns
questions = raw_headers.iloc[0].replace("", pd.NA).ffill()
options = raw_headers.iloc[1]
print(questions, options)

In [ ]:
# Make unique names from the two header rows
names = []
for q, opt in zip(questions, options):
    if opt == "Response":
        names.append(q.strip())
    else:
        names.append(f"{str(q).strip()} -- {opt.strip()}")

In [ ]:

df_2011 = pd.read_csv(
    path, header=None, skiprows=2, names=names,
    encoding="cp1252", low_memory=False)

In [ ]:
print(df_2011.shape)
display(df_2011.head())

In [ ]:
def combine_options(df, start, end):
    """Return 'Option A; Option B' for selected option columns."""
    return (
        df.iloc[:, start:end + 1]
        .apply(
            lambda row: "; ".join(
                [str(x).strip() for x in row if pd.notna(x) and str(x).strip()]
            ),
            axis=1
        )
        .replace("", pd.NA)
    )

multi_groups = {
    "Purchasing involvement": (8, 14),
    "Purchase types": (15, 20),
    "Company budget range": (21, 28),
    "Languages proficient in": (30, 42),
    "Technology products owned": (46, 62),
}

for name, (start, end) in multi_groups.items():
    df_2011[name] = combine_options(df_2011, start, end)

salary_col = "Including bonus, what is your annual compensation in USD?"
print(df_2011[salary_col].value_counts(dropna=False))

In [ ]:
df_2011

In [ ]:
# 1. Inspect the salary answers
salary_col = "Including bonus, what is your annual compensation in USD?"
print("=== Salary value counts ===")
print(df_2011[salary_col].value_counts(dropna=False))

# 2. Create a numeric salary target (ordinal encoding of ranges)
def salary_to_numeric(val):
    if pd.isna(val):
        return pd.NA
    val = str(val).strip()
    if val == "Student / Unemployed":
        return 0
    if val.startswith("<$10,000"):
        return 5_000
    if val.startswith("$10,001 - $25,000"):
        return 17_500
    if val.startswith("$25,001 - $40,000"):
        return 32_500
    if val.startswith("$41,000 - $75,000"):
        return 58_000
    if val.startswith("$75,001 - $100,000"):
        return 87_500
    if val.startswith("$100,001 - $150,000"):
        return 125_000
    if val.startswith(">$150,000"):
        return 175_000  # placeholder
    return pd.NA

df_2011["SalaryNumeric"] = df_2011[salary_col].apply(salary_to_numeric)

# 3. Build a feature set for modeling
features = {
    "Country": "What Country or Region do you live in?",
    "Age": "How old are you?",
    "YearsCoding": "How many years of IT/Programming experience do you have?",
    "Industry": "How would you best describe the industry you work in?",
    "CompanySize": "Which best describes the size of your company?",
    "Occupation": "Which of the following best describes your occupation?",
    "OS": "What operating system do you use the most?",
    "JobSatisfaction": "Please rate your job/career satisfaction",
    "Languages": "Languages proficient in",  # our combined column
}

df_sample = pd.DataFrame()
for short_name, long_name in features.items():
    df_sample[short_name] = df_2011[long_name]

df_sample["SalaryNumeric"] = df_2011["SalaryNumeric"]

# 4. Inspect the sample
print("\n=== Sample shape ===")
print(df_sample.shape)
print("\n=== Missing values ===")
print(df_sample.isna().sum())
print("\n=== First 5 rows ===")
display(df_sample.head())
print("\n=== Salary distribution ===")
print(df_sample["SalaryNumeric"].value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# --- 1. Drop the raw option columns (we already combined them) ---
cols_to_drop = [c for c in df_2011.columns if " -- " in c]
df_plot = df_2011.drop(columns=cols_to_drop)

# --- 2. Categorize columns ---
categorical_cols = []
numeric_like_cols = []

for col in df_plot.columns:
    if col == "Languages proficient in" or col == "Technology products owned":
        continue  # skip multi-select, handled separately

    s = df_plot[col]
    n_unique = s.nunique(dropna=True)
    n_missing = s.isna().sum()
    pct = n_missing / len(s)

    if pct > 0.9:
        continue  # skip columns that are almost entirely missing

    if n_unique <= 15:
        categorical_cols.append(col)
    elif pd.to_numeric(s, errors="coerce").notna().mean() > 0.7:
        numeric_like_cols.append(col)

print(f"Categorical columns ({len(categorical_cols)}):")
for c in categorical_cols:
    print(f"  {c}: {df_plot[c].nunique(dropna=True)} unique values")

print(f"\nNumeric-like columns ({len(numeric_like_cols)}):")
for c in numeric_like_cols:
    print(f"  {c}")

# --- 3. Plot distributions ---
n_cat = len(categorical_cols)
n_num = len(numeric_like_cols)

# Plot categorical (top 10 by cardinality to avoid too many)
top_cat = categorical_cols[:10]
n_top_cat = len(top_cat)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(top_cat):
    s = df_plot[col].dropna()
    value_counts = s.value_counts()
    if len(value_counts) > 10:
        top_vals = value_counts.head(10)
        top_vals.plot(kind='bar', ax=axes[i], title=f"{col[:50]}\n(top 10)")
    else:
        value_counts.plot(kind='bar', ax=axes[i], title=f"{col[:50]}\n(all {len(value_counts)} values)")
    axes[i].tick_params(axis='x', rotation=45)

# Hide unused subplots
for j in range(i+1, n_top_cat):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("2011_categorical_distributions.png", dpi=150, bbox_inches='tight')
plt.show()

# Plot numeric-like
if numeric_like_cols:
    fig, axes = plt.subplots(1, len(numeric_like_cols), figsize=(5*len(numeric_like_cols), 5))
    if len(numeric_like_cols) == 1:
        axes = [axes]
    for i, col in enumerate(numeric_like_cols):
        s = pd.to_numeric(df_plot[col], errors="coerce").dropna()
        if len(s) > 0:
            s.plot(kind='hist', bins=30, ax=axes[i], title=f"{col[:50]}")
    plt.tight_layout()
    plt.savefig("2011_numeric_distributions.png", dpi=150, bbox_inches='tight')
    plt.show()

# --- 4. Multi-select column distributions ---
multi_cols = ["Languages proficient in", "Technology products owned", "Purchasing involvement", "Purchase types", "Company budget range"]
valid_multi = [c for c in multi_cols if c in df_plot.columns]

if valid_multi:
    n_multi = len(valid_multi)
    fig, axes = plt.subplots(1, n_multi, figsize=(6*n_multi, 5))
    if n_multi == 1:
        axes = [axes]
    for i, col in enumerate(valid_multi):
        s = df_plot[col].dropna()
        # For multi-select, explode and count
        exploded = s.str.split("; ").explode()
        exploded = exploded[exploded != ""]
        top = exploded.value_counts().head(15)
        top.plot(kind='bar', ax=axes[i], title=col[:50])
        axes[i].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig("2011_multi_select_distributions.png", dpi=150, bbox_inches='tight')
    plt.show()

# --- 5. Correlation matrix ---
# Only for numeric-like columns (and SalaryNumeric if it exists)
numeric_for_corr = []
for col in df_plot.columns:
    if col in numeric_like_cols or col == "SalaryNumeric":
        s = pd.to_numeric(df_plot[col], errors="coerce")
        if s.notna().sum() > 10:  # at least 10 valid values
            numeric_for_corr.append(col)

if numeric_for_corr:
    corr_df = df_plot[numeric_for_corr].apply(pd.to_numeric, errors="coerce")
    corr = corr_df.corr()

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                square=True, linewidths=0.5)
    plt.title("Correlation Matrix (Numeric Columns)")
    plt.tight_layout()
    plt.savefig("2011_correlation_matrix.png", dpi=150, bbox_inches='tight')
    plt.show()

    print("\n=== Strongest correlations with SalaryNumeric ===")
    if "SalaryNumeric" in corr.columns:
        corr_salary = corr["SalaryNumeric"].drop("SalaryNumeric").abs().sort_values(ascending=False)
        print(corr_salary.head(10))

In [ ]:
import pandas as pd

# Use df_2011 (combined columns already added)
cols_to_skip = [c for c in df_2011.columns if " -- " in c]
investigate = [c for c in df_2011.columns if c not in cols_to_skip]

for col in investigate:
    s = df_2011[col].dropna().astype(str).str.strip()
    s = s[s != ""]

    print("=" * 75)
    print(f"COLUMN: {col}")
    print(f"  Non-null responses: {len(s)}   Unique: {s.nunique()}")

    # FULL value counts if small, else top 25 + tail check
    vc = s.value_counts()
    if len(vc) <= 25:
        print("  All values:")
        for v, n in vc.items():
            print(f"    {v!r:60} {n}")
    else:
        print("  Top 25 values:")
        for v, n in vc.head(25).items():
            print(f"    {v!r:60} {n}")
        print(f"  ... plus {len(vc)-25} rarer values")
        print("  Rarest 10 values:")
        for v, n in vc.tail(10).items():
            print(f"    {v!r:60} {n}")

    # Suspicious numeric-looking entries
    nums = pd.to_numeric(s, errors="coerce").dropna()
    if len(nums):
        print(f"  Numeric-parseable: {len(nums)} values | min={nums.min()} max={nums.max()}")
        weird = nums[(nums > 100) | (nums < 0)]
        if len(weird):
            print(f"  *** SUSPICIOUS numbers (out of plausible range): {sorted(weird.unique())[:20]}")
    print()

In [ ]:
exp_col = "How many years of IT/Programming experience do you have?"

print("dtype:", df_2011[exp_col].dtype)
print("\nRaw value counts:")
print(df_2011[exp_col].value_counts(dropna=False))

print("\nTypes present:")
print(df_2011[exp_col].map(type).value_counts())

In [ ]:
exp_col = "How many years of IT/Programming experience do you have?"

df_2011[exp_col] = df_2011[exp_col].replace({
    "41435": "6-10",
    "41310": "2-5",
})

print("=== After fix ===")
print(df_2011[exp_col].value_counts(dropna=False))

In [ ]:
df_2011.head()

In [ ]:
country_col = "What Country or Region do you live in?"

vals = df_2011[country_col].dropna()
total = len(df_2011)

counts = vals.value_counts()
pcts = counts / total * 100

print("=== Country/Region distribution (2011) ===")
print(f"Total respondents: {total}")
print(f"Non-null: {len(vals)} ({len(vals)/total*100:.1f}%)")
print()
display(pd.DataFrame({
    "Count": counts,
    "Percent": pcts
}).head(30))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(counts.index, counts.values, color='#1a73e8')

# Add percentage labels
for bar, pct in zip(bars, pcts):
    ax.text(
        bar.get_width() + 10,
        bar.get_y() + bar.get_height() / 2,
        f'{pct:.1f}%',
        va='center',
        fontsize=9
    )

ax.set_title('2011 Stack Overflow Survey: Country/Region Distribution\n(Top 20)', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Respondents', fontsize=12)
ax.set_ylabel('Country or Region', fontsize=12)
plt.tight_layout()
plt.savefig('country_distribution_2011.png', dpi=150)
plt.show()

In [ ]:
# Age distribution
age_col = "How old are you?"
vals = df_2011[age_col].dropna()
age_order = ['< 20', '20-24', '25-29', '30-34', '35-39', '40-50', '51-60', '>60']
counts = vals.value_counts().reindex(age_order).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(counts.index, counts.values, color='#0f9d58')
for bar, count in zip(ax.patches, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), str(int(count)), ha='center', va='bottom', fontsize=9)
ax.set_title('2011: Age Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Respondents')
plt.tight_layout()
plt.savefig('age_distribution_2011.png', dpi=150)
plt

In [ ]:
# Years Coding distribution
exp_col = "How many years of IT/Programming experience do you have?"
exp_order = ["<2", "2-5", "6-10", "11+"]
vals = df_2011[exp_col].dropna()
counts = vals.value_counts().reindex(exp_order).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(counts.index, counts.values, color='#f4b400')
ax.set_title('2011: Years of IT/Programming Experience', fontsize=14, fontweight='bold')
ax.set_xlabel('Years Coding')
ax.set_ylabel('Respondents')
plt.tight_layout()
plt.savefig('experience_distribution_2011.png', dpi=150)
plt.show()

In [ ]:
# Salary distribution
salary_col = "Including bonus, what is your annual compensation in USD?"
vals = df_2011[salary_col].dropna()
salary_order = ['Student / Unemployed', '<$20,000', '$20,000 - $40,000',
                '$40,000 - $60,000', '$60,000 - $80,000', '$80,000 - $100,000',
                '$100,000 - $120,000', '$120,000 - $140,000', '>$140,000' ]


counts = vals.value_counts().reindex(salary_order).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(counts.index, counts.values, color='#db4437')
ax.set_title('2011: Annual Compensation Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Salary Range')
ax.set_ylabel('Respondents')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('salary_distribution_2011.png', dpi=150)
plt.show()

In [ ]:
# Occupation distribution
occ_col = "Which of the following best describes your occupation?"
vals = df_2011[occ_col].dropna()
counts = vals.value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(counts.index, counts.values, color='#4285f4')
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height() / 2, str(int(count)), va='center', fontsize=9)
ax.set_title('2011: Occupation Distribution (Top 15)', fontsize=14, fontweight='bold')
ax.set_xlabel('Respondents')
ax.set_ylabel('Occupation')
plt.tight_layout()
plt.savefig('occupation_distribution_2011.png', dpi=150)
plt.show()

In [ ]:
# OS distribution
os_col = "What operating system do you use the most?"
vals = df_2011[os_col].dropna()
counts = vals.value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(counts.index, counts.values, color='#9c27b0')
ax.set_title('2011: Primary Operating System', fontsize=14, fontweight='bold')
ax.set_xlabel('Operating System')
ax.set_ylabel('Respondents')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('os_distribution_2011.png', dpi=150)
plt.show()

In [ ]:
# Job Satisfaction distribution
sat_col = "Please rate your job/career satisfaction"
vals = df_2011[sat_col].dropna()
sat_order = ["FML", "I'm not happy in my job", "It pays the bills", "I enjoy going to work", "So happy it hurts"]
counts = vals.value_counts().reindex(sat_order).fillna(0)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(counts.index, counts.values, color='#00bcd4')
ax.set_title('2011: Job/Career Satisfaction', fontsize=14, fontweight='bold')
ax.set_xlabel('Satisfaction Level')
ax.set_ylabel('Respondents')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('satisfaction_distribution_2011.png', dpi=150)
plt.show()



In [ ]:
# Languages proficient in (multi-select, so we need to explode)
lang_col = "Languages proficient in"
vals = df_2011[lang_col].dropna()
lang_counts = vals.str.split(';').explode().value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(lang_counts.index, lang_counts.values, color='#ff5722')
ax.set_title('2011: Languages Proficient', fontsize=14, fontweight='bold')
ax.set_xlabel('Language')
ax.set_ylabel('Number of Developers')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('languages_distribution_2011.png', dpi=150)
plt.show()

In [ ]:
lang_col = "Languages proficient in"
vals = df_2011[lang_col].dropna()

# Show which languages appear and how many developers selected each
lang_counts = vals.str.split(';').explode().value_counts()

print("=== Languages Proficient In ===")
print(lang_counts)
print(f"\nTotal developers who selected at least one language: {len(vals)}")
print(f"Unique languages: {len(lang_counts)}")

# Also show respondents who selected "None"
none_count = (vals == "None").sum()
print(f"\nRespondents who selected 'None': {none_count} ({none_count/len(vals)*100:.1f}%)")

# Show respondents who selected "other (please specify)"
other_count = (vals == "other (please specify)").sum()
print(f"Respondents who selected 'other (please specify)': {other_count}")

# If there are "other" responses, show what they typed
other_responses = vals[vals.str.contains("other (please specify)", case=False, na=False)]
if len(other_responses) > 0:
    print("\n=== Other (please specify) responses ===")
    print(other_responses.value_counts())

In [ ]:
other_col = df_2011.columns[42]   # "Which languages are you proficient in? -- other (please specify)"
print("Column name:", other_col)

free_text = df_2011[other_col].dropna()
free_text = free_text[free_text.astype(str).str.strip() != ""]

print(f"\nFree-text responses: {len(free_text)}")
print("\n=== Raw values ===")
display(free_text.value_counts().head(50))

In [ ]:
import re

def extract_language(text):
    t = str(text).strip()

    # Drop commentary that isn't a language name
    if re.search(r"^(none|no|n/?a|other|nothing)\b", t, re.I):
        return pd.NA

    # Strip common filler
    t = re.sub(r"\b(i\s+(use|know|work with|program in)|also|but|i'd|rather|mainly|mostly|some|a\s+bit\s+of|etc\.?)\b",
               "", t, flags=re.I)
    t = re.sub(r"[.!?]+$", "", t).strip(" ,;:")

    # Split anything listing multiple languages
    parts = [p.strip() for p in re.split(r"[,/&+]|\band\b", t, flags=re.I) if p.strip()]
    return "; ".join(parts) if parts else pd.NA

clean_other = free_text.map(extract_language).dropna()

print("=== Cleaned 'other' languages ===")
print(clean_other.str.split(";").explode().str.strip().value_counts().head(40))


In [ ]:
lang_col = "Languages proficient in"

df_2011["_other_clean"] = pd.NA
df_2011.loc[clean_other.index, "_other_clean"] = clean_other

# Append cleaned free text onto the existing multi-select string
mask = df_2011["_other_clean"].notna() & df_2011[lang_col].notna()
df_2011.loc[mask, lang_col] = df_2011.loc[mask, lang_col] + "; " + df_2011.loc[mask, "_other_clean"]

mask_only = df_2011["_other_clean"].notna() & df_2011[lang_col].isna()
df_2011.loc[mask_only, lang_col] = df_2011.loc[mask_only, "_other_clean"]

df_2011.drop(columns="_other_clean", inplace=True)

# Final tally
all_langs = df_2011[lang_col].dropna().str.split(";").explode().str.strip()
all_langs = all_langs[(all_langs != "") & (all_langs.str.lower() != "none")]
print(all_langs.value_counts())

In [ ]:
lang_col = "Languages proficient in"

# Standardize casing
aliases = {
    "javascript": "JavaScript",
    "js": "JavaScript",
    "c++": "C++",
    "objective-c": "Objective-C",
    "c": "C",
    "c#": "C#",
    "css": "CSS",
    "php": "PHP",
    "python": "Python",
    "ruby": "Ruby",
    "sql": "SQL",
    "perl": "Perl",
    "java": "Java",
}

def standardize_language(text):
    return aliases.get(text.lower().strip(), text)

def clean_languages(text):
    # Split on semicolons, commas, slashes, ampersands, plus signs
    parts = [p.strip() for p in re.split(r"[,;/&+]", str(text)) if p.strip()]
    parts = [standardize_language(p) for p in parts if p.lower() != "none"]
    return "; ".join(parts) if parts else pd.NA

df_2011[lang_col] = df_2011[lang_col].apply(clean_languages)

# Final tally
all_langs = df_2011[lang_col].dropna().str.split(";").explode().str.strip()
all_langs = all_langs[(all_langs != "") & (all_langs.str.lower() != "none")]

print("=== Final Standardized Languages ===")
print(all_langs.value_counts())

In [ ]:
known_languages = [
        "C", "C#", "C++", "CSS", "CoffeeScript", "Dart", "DM", "Elixir", "Go", "Groovy",
        "HTML", "Java", "JavaScript", "Kotlin", "Objective-C", "Perl", "PHP", "PowerShell",
        "Python", "Ruby", "Rust", "Scala", "Shell", "Swift", "TypeScript", "1C Enterprise",
        "4D", "ABAP", "ActionScript", "Ada", "Agda", "AL", "AMPL", "AngelScript", "ANTLR",
        "APL", "Apollo Guidance Computer", "Arduino", "ASL", "ASP.NET", "AspectJ", "Assembly",
        "ATS", "AutoHotkey", "AutoIt", "Awk", "Ballerina", "Batchfile", "Befunge", "BibTeX",
        "Bison", "BlitzBasic", "BlitzMax", "Bluespec", "Boo", "Boogie", "Brainfuck", "Brightscript",
        "C-ObjDump", "C2hs Haskell", "Cap'n Proto", "CartoCSS", "Ceylon", "Chapel", "Charity",
        "ChucK", "Cirru", "Clarion", "Clean", "Click", "CLIPS", "Clojure", "CMake", "COBOL",
        "CoffeeScript", "ColdFusion", "ColdFusion CFC", "Common Lisp", "Component Pascal",
        "Cool", "Coq", "Coral", "Csound", "Csound Document", "Csound Score", "CSS", "Cuda",
        "CWeb", "Cycript", "Cython", "D", "Dafny", "Darcs Patch", "Dart", "DataWeave",
        "Delphi", "DIGITAL Command Language", "DM", "DNS Zone", "Doc comments", "DocRaptor",
        "DCL", "Dockerfile", "Dogescript", "DTrace", "Dylan", "E", "ECL", "ECLiPSe", "Eiffel",
        "EJS", "Elixir", "Elm", "Emacs Lisp", "EmberScript", "Erlang", "Euphoria",
        "F#", "F*","FIGlet Font", "FIRRTL", "Forth", "Fortran", "FreeMarker", "Frege",
        "Forth", "Forth", "G-code", "Game Maker Language", "GAMS", "GAP", "Gate", "GAS",
        "GCC Machine Description", "GDB", "GDScript", "Genie", "Genshi", "Gentoo Ebuild",
        "Gentoo Eclass", "Gherkin", "GLSL", "Glyph", "Gnuplot", "Go", "Golo", "Gosu", "Grace",
        "Grammatical Framework", "GraphQL", "Groovy", "Groovy Server Pages", "Hack", "Hack",
        "HTML", "HTML+Django", "HTML+ECR", "HTML+EEX", "HTML+ERB", "HTML+PHP", "HTML+Razor",
        "HTTP", "HXML", "Haskell", "Haxe", "Hy", "HyPhy", "IDL", "IGOR Pro", "INI", "Inno",
        "IRC log", "Idris", "Inform 7", "Ink", "Ink", "Isabelle", "Isabelle ROOT", "J",
        "JFlex", "Jison", "Jison Lex", "Jolie", "JSON", "JSON5", "JSONiq", "JSONld", "JSX",
        "Jade", "Janet", "Java", "Java Properties", "Java Server Pages", "JavaScript",
        "JavaScript+ERB", "Jest", "Jison", "Jolie", "JSON", "JSON5", "JSONiq", "Jsonnet",
        "Jupyter Notebook", "K", "Kconfig", "Kemal", "Kit", "Kotlin", "KRL", "LabVIEW",
        "Lasso", "Latte", "Lean", "Lex", "LFE", "LilyPond", "Limbo", "Lingo", "Liquid",
        "Literate Agda", "Literate CoffeeScript", "Literate Haskell", "LiveScript", "LLVM",
        "Logos", "Logtalk", "LOLCODE", "LookML", "LoomScript", "Lua", "Lucene", "Luduscript",
        "Lutin", "LynxScript", "M", "M4Sugar", "M4", "MACRO 11", "Makefile", "Mako", "Markdown",
        "Marko", "Mask", "Mathematica", "Matlab", "Mako", "Max", "Max/MSP", "Mercury",
        "Metal", "MiniD", "Mirah", "Model 204", "Mole", "Monkey", "Moocode", "MoonScript",
        "MQL4", "MQL5", "MTML", "MUF", "Muf", "mupad", "Myghty", "NCL", "NEON", "NCL",
        "NL", "NCL", "NSIS", "Nasal", "Nemerle", "NetLinx", "NetLinx+ERB", "NetLogo",
        "NewLisp", "Nextflow", "Nginx", "Nim", "Nit", "Nix", "Nu", "NumPy", "Nunjucks",
        "OASys", "OCaml", "Omgrofl", "ooc", "Opa", "Opal", "OpenCL", "OpenEdge ABL",
        "OpenQASM", "OpenRC runscript", "OpenSCAD", "Org", "Oz", "P4", "P4", "PDDL",
        "PEG.js", "PHP", "PLSQL", "PLpgSQL", "PogoScript", "Pony", "Portugol", "PostCSS",
        "PostScript", "POV-Ray SDL", "PowerShell", "Prisma", "Processing", "Prolog",
        "Propeller Spin", "Puppet", "Pyret", "Python", "Python traceback", "Q", "QMake",
        "QML", "R", "RAML", "Rascal", "RDoc", "REALbasic", "Reason", "Rebol", "Red",
        "Redcode", "RenderScript", "ReScript", "Rexx", "Ring", "RobotFramework", "Roff",
        "Rouge", "RPC", "RPC", "Ruby", "Rust", "SAS", "SCSS", "SQF", "SQL", "SQLPL",
        "SRecode Template", "SVG", "Sage", "SaltStack", "Sas", "Sass", "Scala", "Scaml",
        "Scenic", "Scheme", "Scilab", "SCSS", "Self", "ShaderLab", "Shell", "ShellSession",
        "Shen", "Slash", "Slice", "Slim", "Smali", "Smalltalk", "Smarty", "Solidity",
        "SourcePawn", "Squirrel", "SRecode Template", "Stan", "Standard ML", "Stata",
        "Stylus", "SuperCollider", "Svelte", "Swift", "SystemVerilog", "TI Program",
        "TLA", "TSQL", "TSX", "TXL", "Tcl", "Tcsh", "Terra", "TeX", "Tea", "Text",
        "Textile", "Thrift", "TI Program", "TLA", "TOML", "TSQL", "TSX", "TXL", "Tcl",
        "Tcsh", "Terra", "TeX", "Tea", "Text", "Textile", "Thrift", "TOML", "Transact-SQL",
        "Turing", "Turtle", "Twig", "TXL", "TypeScript", "TypeScript JSX", "Typst",
        "Unified Parallel C", "Unity3D Asset", "Unity3D Asset", "Unix Assembly", "Uno",
        "UnrealScript", "UrWeb", "Vala", "VBA", "VBScript", "VCL", "VCL", "Velocity",
        "Verilog", "VHDL", "Vim script", "Visual Basic .NET", "Volt", "Vue", "Vyper",
        "WDL", "WebIDL", "WebIDL", "Wget", "Wikipedia", "Wollok", "Wolfram Language",
        "X10", "xBase", "XC", "Xtend", "Xojo", "XProc", "XQuery", "XS", "XSL", "Xtend",
        "Xtend", "YAML", "YANG", "Yacc", "YAML", "YANG", "Yacc", "ZAP", "Zeek", "ZenCode",
        "Zeek", "Zephir", "Zig", "ZIL", "Zustand", "Zig"]

all_langs = df_2011[lang_col].dropna().str.split(";").explode().str.strip()
all_langs = all_langs[(all_langs != "") & (all_langs.str.lower() != "none")]

# Find languages NOT in known set
unknown = all_langs[~all_langs.isin(known_languages)]

print("=== Unknown Languages (not in known_languages) ===")
if len(unknown) > 0:
    for lang, count in unknown.value_counts().items():
        print(f"  '{lang}': {count}")
else:
    print("  All languages are accounted for in known_languages.")

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")
YEARS = range(2011, 2027)

schema = {}

ENCODINGS = ["utf-8-sig", "cp1252", "latin1"]

for year in YEARS:
    file_path = DATA_DIR / str(year) / "results.csv"

    if not file_path.exists():
        print(f"Missing: {year}")
        continue

    # 2015 has an extra junk row above the real header, so read the 2nd row instead
    skiprows = [0] if year == 2015 else None

    for encoding in ENCODINGS:
        try:
            columns = pd.read_csv(
                file_path,
                nrows=0,
                skiprows=skiprows,
                encoding=encoding
            ).columns.tolist()

            schema[str(year)] = pd.Series(columns)

            print(f"{year}: {len(columns)} questions")
            break

        except UnicodeDecodeError:
            continue

# Combine into a wide DataFrame
schema_df = pd.DataFrame(schema)

# Save
schema_df.to_csv(
    "survey_schema_by_year.csv",
    index=False,
    encoding="utf-8-sig"
)

print(schema_df.shape)
display(schema_df)

In [ ]:
schema_df

In [ ]:
xxfrom pathlib import Path
import pandas as pd

DATA_DIR = Path("data")
YEARS = range(2011, 2027)

ENCODINGS = ["utf-8-sig", "cp1252", "latin1"]

def read_metadata(path):
    """Return (questions, options, encoding, header_row) for a year's CSV."""
    for enc in ENCODINGS:
        try:
            raw = pd.read_csv(
                path, header=None, nrows=3,
                encoding=enc, dtype=str, keep_default_na=False
            )
        except UnicodeDecodeError:
            continue

        row0 = raw.iloc[0].fillna("").astype(str).str.strip()
        nonempty_0 = int((row0 != "").sum())
        nonempty_1 = int((raw.iloc[1].fillna("").astype(str).str.strip() != "").sum()) if len(raw) > 1 else 0

        # Merged-header layout: question text on one row, option labels below it.
        # True for 2011-2014 and for 2015 (header at row 1).
        merged = nonempty_0 < nonempty_1 * 0.7 or (year == 2015)
        header_row = 1 if merged else 0

        questions = raw.iloc[header_row].replace("", pd.NA).ffill()
        next_idx = header_row + 1
        options = (
            raw.iloc[next_idx]
            if len(raw) > next_idx
            else pd.Series([""] * len(questions))
        )
        return questions, options, enc, header_row

    return None, None, None, None

per_year = {}

for year in YEARS:
    path = DATA_DIR / str(year) / "results.csv"   # you renamed the 2026 file too

    if not path.exists():
        print(f"{year}: MISSING")
        continue

    questions, options, enc, header_row = read_metadata(path)
    if questions is None:
        print(f"{year}: could not read")
        continue

    # Build a readable label per column: "Question || option"
    labels = []
    for q, opt in zip(questions, options):
        q = "" if pd.isna(q) else str(q).strip()
        opt = "" if pd.isna(opt) else str(opt).strip()
        labels.append(f"{q} || {opt}" if opt and opt.lower() != "response" else q)

    per_year[str(year)] = labels
    print(f"{year}: {len(labels)} columns | header row {header_row} | {enc}")

# Pad shorter years with NaN so all lists align by position
width = max(len(v) for v in per_year.values())
padded = {y: v + [None] * (width - len(v)) for y, v in per_year.items()}

schema_df = pd.DataFrame(padded)
schema_df.index.name = "col_position"

schema_df.to_csv("survey_schema_by_year.csv", index=True, encoding="utf-8-sig")

print("\nShape:", schema_df.shape)
display(schema_df.head(15))